In [ ]:
# MODEL TRAINING

import re
from typing import Tuple
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    # url
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # de-identification
    text = re.sub(r'@\S+', '', text)
    # removal of hashtag
    text = text.replace('#', '')
    # normalization
    text = text.lower().strip()
    return text

class PHTweetElectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# Model Architecture
class XLMRobertaPureClassifier(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_classes=3):
        super(XLMRobertaPureClassifier, self).__init__()
        # Load core transformer backbone
        self.xlm_roberta = AutoModel.from_pretrained(model_name)

        # Setup for dropout and type of classifier
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        # Extract sequence outputs from transformer
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use CLS token representation (first token in sequence)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Apply regularizations and classify
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

# Training Pipeline
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    df = pd.read_csv("/kaggle/input/datasets/andryjosephtumacole/masked/2022_combined_coarse_masked.csv")
    
    # Preprocess tweets
    df['text'] = df['text'].apply(preprocess_tweet)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    df = df[df['text'] != ""].reset_index(drop=True)

    # Encode target labels numerically
    label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    df['label'] = df['sentiment'].map(label_mapping)
    
    # 80-20 train test split
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['text'].values, df['label'].values, train_size=0.8, test_size=0.2, random_state=42
    )

    model_name = 'xlm-roberta-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = XLMRobertaPureClassifier(model_name=model_name, num_classes=3)
    model.to(device)

    # Processes the tweets with the tokenizer
    train_dataset = PHTweetElectionDataset(train_texts, train_labels, tokenizer)
    test_dataset = PHTweetElectionDataset(test_texts, test_labels, tokenizer)

    # Readies the data for feeding to the model
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    # Balanced learning rates tailored for textual fine-tuning
    backbone_params = list(model.xlm_roberta.parameters())
    head_params = list(model.classifier.parameters())

    optimizer = AdamW([
        {'params': backbone_params, 'lr': 2e-5}, # Conservative updates for transformer backbone
        {'params': head_params, 'lr': 1e-4}      # Focused, stable updates for the linear head
    ], weight_decay=0.01)

    epochs = 4
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    # Calculate balanced weights to handle class imbalances
    raw_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    print(f"Injected class weights successfully: {raw_weights}")

    print("Beginning Training Phase...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    print("Evaluating Performance on Test Set...")
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1).flatten().cpu().numpy()
            
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, predictions)
    print(f"Model Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions, target_names=['Negative', 'Neutral', 'Positive']))

    torch.save(model.state_dict(), "xlmr_pure_model.pt")
    print("Model assets stored successfully for future inference pipelines.")

if __name__ == "__main__":
    main()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Injected class weights successfully: [0.96615753 0.89103715 1.18668383]
Beginning Training Phase...
Epoch 1/4 | Loss: 0.8620
Epoch 2/4 | Loss: 0.6603
Epoch 3/4 | Loss: 0.5612
Epoch 4/4 | Loss: 0.4747
Evaluating Performance on Test Set...
Model Accuracy: 0.7336

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.86      0.84      1681
     Neutral       0.73      0.67      0.70      1838
    Positive       0.63      0.67      0.65      1320

    accuracy                           0.73      4839
   macro avg       0.73      0.73      0.73      4839
weighted avg       0.73      0.73      0.73      4839

Model assets stored successfully for future inference pipelines.


In [ ]:
# MODEL TESTING

import re
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '')  
    text = text.lower().strip()
    return text

class XLMRobertaPureClassifier(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_classes=3):
        super(XLMRobertaPureClassifier, self).__init__()
        self.xlm_roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = XLMRobertaPureClassifier(model_name=model_name, num_classes=3)
model.load_state_dict(torch.load("xlmr_pure_model.pt", map_location=device))
model.to(device)
model.eval()  

def predict_tweets(tweet_list):
    label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    results = []
    
    for raw_tweet in tweet_list:
        # Preprocess using the identical pipeline
        cleaned_tweet = preprocess_tweet(raw_tweet)
        
        # Tokenize
        encoding = tokenizer(
            cleaned_tweet,
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        # Run prediction without tracking gradients
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            probabilities = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(logits, dim=1).item()
            confidence = probabilities[0][pred_class].item()
            
        results.append({
            "raw": raw_tweet,
            "prediction": label_mapping[pred_class],
            "confidence": f"{confidence * 100:.2f}%"
        })
        
    return results

sample_tweets = [
    "Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸",
    "Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.",
    "Abangan na lang natin ang official count at huwag mag-away away.",
    "BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident",
    "I'm still undecided on who to vote for, need to read more platforms.",
    'Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble Islam faith. His actions do not speak of the sentiments, beliefs and behavior of peace-loving Muslims. His religion should not be an issue here, HIS BAD ATTITUDE AND POLITICAL DECISIONS ARE!',
    "MABUTI NGA SA ABS-CBN.  Sobrang mukhang pera talaga yang business establishment na yan."
]

print("\nRunning Inference on Sample Tweets...")
predictions = predict_tweets(sample_tweets)

for res in predictions:
    print(f"\nTweet: {res['raw']}")
    print(f"Predicted Sentiment: {res['prediction']} ({res['confidence']} confidence)")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Inference on Sample Tweets...

Tweet: Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸
Predicted Sentiment: Positive (93.98% confidence)

Tweet: Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.
Predicted Sentiment: Negative (95.38% confidence)

Tweet: Abangan na lang natin ang official count at huwag mag-away away.
Predicted Sentiment: Neutral (95.25% confidence)

Tweet: BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident
Predicted Sentiment: Positive (96.53% confidence)

Tweet: I'm still undecided on who to vote for, need to read more platforms.
Predicted Sentiment: Neutral (92.54% confidence)

Tweet: Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble Is

In [ ]:
# PSEUDO-LABELING

import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm 

def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '')
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    
    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025 +  CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class XLMRobertaPureClassifier(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_classes=3):
        super(XLMRobertaPureClassifier, self).__init__()
        self.xlm_roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

class PseudoLabelDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    df_2025 = pd.read_csv("/kaggle/input/datasets/andryjosephtumacole/latest/2025.csv")
    
    print(f"Loaded 2025 dataset with {len(df_2025)} rows.")
    
    print("Preprocessing text...")
    df_2025['cleaned_text'] = df_2025['text'].apply(preprocess_tweet)

    # Initialize model and load weights
    model_name = 'xlm-roberta-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = XLMRobertaPureClassifier(model_name=model_name, num_classes=3)
    model.load_state_dict(torch.load("xlmr_pure_model.pt", map_location=device))
    model.to(device)
    model.eval()

    # Optimized DataLoader for fast GPU processing
    inference_dataset = PseudoLabelDataset(df_2025['cleaned_text'].values, tokenizer)
    inference_loader = DataLoader(inference_dataset, batch_size=64, shuffle=False, num_workers=2)

    all_preds = []
    all_confidences = []

    print("Running batched pseudo-labeling inference...")
    with torch.no_grad():
        for batch in tqdm(inference_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids, attention_mask)
            probabilities = torch.softmax(logits, dim=1)
            
            # Get max probability (confidence) and the corresponding class index
            confidences, preds = torch.max(probabilities, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

    # Map indices back to labels
    label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    df_2025['pseudo_label'] = [label_mapping[p] for p in all_preds]
    df_2025['confidence'] = all_confidences

    print("\n" + "="*50)
    print("Raw Pseudo-Label Distribution (Before Filtering)")
    print("="*50)
    raw_counts = df_2025['pseudo_label'].value_counts()
    raw_percents = df_2025['pseudo_label'].value_counts(normalize=True) * 100
    for idx in raw_counts.index:
        print(f"{idx}: {raw_counts[idx]} ({raw_percents[idx]:.2f}%)")

    # Only keep rows with 80% confidence
    confidence_threshold = 0.80
    df_filtered = df_2025[df_2025['confidence'] >= confidence_threshold].copy()

    print("\n" + "="*50)
    print(f"FILTERED DISTRIBUTION (CONFIDENCE >= {confidence_threshold*100}%)")
    print("="*50)
    print(f"Retained Rows: {len(df_filtered)} out of {len(df_2025)} ({len(df_filtered)/len(df_2025)*100:.2f}%)")
    
    if len(df_filtered) > 0:
        filt_counts = df_filtered['pseudo_label'].value_counts()
        filt_percents = df_filtered['pseudo_label'].value_counts(normalize=True) * 100
        for idx in filt_counts.index:
            print(f"{idx}: {filt_counts[idx]} ({filt_percents[idx]:.2f}%)")
    else:
        print("No rows were above the threshold")

    df_filtered.to_csv("2025_pseudo_labeled_filtered.csv", index=False)
    print("\nFiltered pseudo-labeled asset saved as '2025_pseudo_labeled_filtered.csv'")

if __name__ == "__main__":
    main()

Using device: cuda
Loaded 2025 dataset with 217640 rows.
Preprocessing text...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running batched pseudo-labeling inference...


100%|██████████| 3401/3401 [27:38<00:00,  2.05it/s]



Raw Pseudo-Label Distribution (Before Filtering)
Negative: 96524 (44.35%)
Neutral: 77594 (35.65%)
Positive: 43522 (20.00%)

FILTERED DISTRIBUTION (CONFIDENCE >= 80.0%)
Retained Rows: 133634 out of 217640 (61.40%)
Negative: 67172 (50.27%)
Neutral: 45655 (34.16%)
Positive: 20807 (15.57%)

Filtered pseudo-labeled asset saved as '2025_pseudo_labeled_filtered.csv'


In [10]:
# DOWNSAMPLING TO BALANCE THE DATASET

import pandas as pd

df_filtered = pd.read_csv("2025_pseudo_labeled_filtered.csv")

# Get the least distribution among the classes
target_size = df_filtered['pseudo_label'].value_counts().min()
print(f"Target size per class determined by minority class: {target_size}")

# Sample each class with the least distribution size to balance the dataset
df_balanced_2025 = (
    df_filtered.groupby('pseudo_label', group_keys=False)
    .apply(lambda x: x.sample(n=target_size, random_state=42))
)

print("Balanced Distribution:\n")
print(df_balanced_2025['pseudo_label'].value_counts())
print(f"Total rows in balanced 2025 dataset: {len(df_balanced_2025)}")

df_balanced_2025.to_csv("2025_pseudo_labeled_balanced.csv", index=False)
print("\nAsset successfully stored as '2025_pseudo_labeled_balanced.csv'")

Target size per class determined by minority class: 20807
Balanced Distribution:

pseudo_label
Negative    20807
Neutral     20807
Positive    20807
Name: count, dtype: int64
Total rows in balanced 2025 dataset: 62421


/tmp/ipykernel_57/71662904.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=target_size, random_state=42))



Asset successfully stored as '2025_pseudo_labeled_balanced.csv'


In [11]:
# MODEL TRAINING FOR PSEUDO-LABELED DATASET

import re
from typing import Tuple
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '')
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    
    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    CANDIDATES_LIST = " ".join(SENATORS_2025 + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class PHTweetElectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

class XLMRobertaPureClassifier(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_classes=3):
        super(XLMRobertaPureClassifier, self).__init__()
        self.xlm_roberta = AutoModel.from_pretrained(model_name)
        
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        # Extract sequence outputs from transformer
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use CLS token representation (first token in sequence)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Apply regularizations and classify
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    df = pd.read_csv("/kaggle/working/2025_pseudo_labeled_balanced.csv")
    
    df['text'] = df['text'].apply(preprocess_tweet)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    df = df[df['text'] != ""].reset_index(drop=True)

    label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    df['label'] = df['pseudo_label'].map(label_mapping)
    
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['text'].values, df['label'].values, train_size=0.8, test_size=0.2, random_state=42
    )

    model_name = 'xlm-roberta-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = XLMRobertaPureClassifier(model_name=model_name, num_classes=3)
    model.to(device)

    train_dataset = PHTweetElectionDataset(train_texts, train_labels, tokenizer)
    test_dataset = PHTweetElectionDataset(test_texts, test_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    backbone_params = list(model.xlm_roberta.parameters())
    head_params = list(model.classifier.parameters())

    optimizer = AdamW([
        {'params': backbone_params, 'lr': 2e-5}, 
        {'params': head_params, 'lr': 1e-4}      
    ], weight_decay=0.01)

    epochs = 3
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    # Calculate balanced weights to handle class imbalances
    raw_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    print(f"Injected class weights successfully: {raw_weights}")

    print("Starting Training Phase...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    # Evaluate performance
    print("Evaluating Performance on Test Set...")
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1).flatten().cpu().numpy()
            
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    # Metrics for analysis
    acc = accuracy_score(true_labels, predictions)
    print(f"\n===== MODEL ACCURACY: {acc:.4f} =====")
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions, target_names=['Negative', 'Neutral', 'Positive']))

    torch.save(model.state_dict(), "xlmr_pseudo_label_model.pt")
    print("Model assets stored successfully for future inference pipelines.")

if __name__ == "__main__":
    main()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Injected class weights successfully: [0.98779925 1.01198553 1.00050813]
Starting Training Phase...
Epoch 1/3 | Loss: 0.3030
Epoch 2/3 | Loss: 0.1233
Epoch 3/3 | Loss: 0.0537
Evaluating Performance on Test Set...

===== MODEL ACCURACY: 0.9842 =====

Classification Report:
              precision    recall  f1-score   support

    Negative       0.98      0.98      0.98      4088
     Neutral       0.99      0.99      0.99      4083
    Positive       0.98      0.99      0.98      4136

    accuracy                           0.98     12307
   macro avg       0.98      0.98      0.98     12307
weighted avg       0.98      0.98      0.98     12307

Model assets stored successfully for future inference pipelines.


In [12]:
# MODEL TESTING FOR PSEUDO-LABELED MODEL

import re
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '')
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025  + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class XLMRobertaPureClassifier(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_classes=3):
        super(XLMRobertaPureClassifier, self).__init__()
        self.xlm_roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.xlm_roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize structure and load saved weights
model = XLMRobertaPureClassifier(model_name=model_name, num_classes=3)
model.load_state_dict(torch.load("xlmr_pseudo_label_model.pt", map_location=device))
model.to(device)
model.eval()  

def predict_tweets(tweet_list):
    label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    results = []
    
    for raw_tweet in tweet_list:
        # Preprocess using the identical pipeline
        cleaned_tweet = preprocess_tweet(raw_tweet)
        
        # Tokenize
        encoding = tokenizer(
            cleaned_tweet,
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        # Run prediction without tracking gradients
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            probabilities = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(logits, dim=1).item()
            confidence = probabilities[0][pred_class].item()
            
        results.append({
            "raw": raw_tweet,
            "prediction": label_mapping[pred_class],
            "confidence": f"{confidence * 100:.2f}%"
        })
        
    return results

sample_tweets = [
    "Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸",
    "Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.",
    "Abangan na lang natin ang official count at huwag mag-away away.",
    "BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident",
    "I'm still undecided on who to vote for, need to read more platforms.",
    'Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble Islam faith. His actions do not speak of the sentiments, beliefs and behavior of peace-loving Muslims. His religion should not be an issue here, HIS BAD ATTITUDE AND POLITICAL DECISIONS ARE!',
    "MABUTI NGA SA ABS-CBN.  Sobrang mukhang pera talaga yang business establishment na yan."
]

print("\nRunning Inference on Sample Tweets...")
predictions = predict_tweets(sample_tweets)

for res in predictions:
    print(f"\nTweet: {res['raw']}")
    print(f"Predicted Sentiment: {res['prediction']} ({res['confidence']} confidence)")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Inference on Sample Tweets...

Tweet: Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸
Predicted Sentiment: Positive (100.00% confidence)

Tweet: Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.
Predicted Sentiment: Positive (91.32% confidence)

Tweet: Abangan na lang natin ang official count at huwag mag-away away.
Predicted Sentiment: Neutral (99.99% confidence)

Tweet: BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident
Predicted Sentiment: Positive (100.00% confidence)

Tweet: I'm still undecided on who to vote for, need to read more platforms.
Predicted Sentiment: Neutral (100.00% confidence)

Tweet: Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble

In [1]:
# EmoBert Training

import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def preprocess_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '') 
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    
    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025  + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class PHTweetElectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class EmoBERTPureClassifier(nn.Module):
    def __init__(self, model_name='bhadresh-savani/bert-base-uncased-emotion', num_classes=3):
        super(EmoBERTPureClassifier, self).__init__()
        self.emobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.emobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.emobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    df = pd.read_csv("/kaggle/input/datasets/andryjosephtumacole/masked/2022_combined_coarse_masked.csv")
    df['text'] = df['text'].apply(preprocess_tweet)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    df = df[df['text'] != ""].reset_index(drop=True)

    label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    df['label'] = df['sentiment'].map(label_mapping)
    
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['text'].values, df['label'].values, train_size=0.8, test_size=0.2, random_state=42
    )

    model_name = 'bhadresh-savani/bert-base-uncased-emotion'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = EmoBERTPureClassifier(model_name=model_name, num_classes=3).to(device)

    train_loader = DataLoader(PHTweetElectionDataset(train_texts, train_labels, tokenizer), batch_size=16, shuffle=True)
    test_loader = DataLoader(PHTweetElectionDataset(test_texts, test_labels, tokenizer), batch_size=16, shuffle=False)

    optimizer = AdamW([
        {'params': model.emobert.parameters(), 'lr': 2e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4}
    ], weight_decay=0.01)

    epochs = 3
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * epochs), num_training_steps=len(train_loader) * epochs)

    class_weights_tensor = torch.tensor(compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = criterion(logits, batch['labels'].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            predictions.extend(torch.argmax(logits, dim=1).flatten().cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

    print(f"\n===== MODEL ACCURACY: {accuracy_score(true_labels, predictions):.4f} =====")
    print(classification_report(true_labels, predictions, target_names=['Negative', 'Neutral', 'Positive']))

    torch.save(model.state_dict(), "emobert_model.pt")
    print("EmoBERT model stored successfully.")

if __name__ == "__main__":
    main()

Using device: cuda


config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3 | Loss: 0.7990
Epoch 2/3 | Loss: 0.6031
Epoch 3/3 | Loss: 0.4558

===== MODEL ACCURACY: 0.7206 =====
              precision    recall  f1-score   support

    Negative       0.81      0.83      0.82      1688
     Neutral       0.73      0.66      0.69      1850
    Positive       0.60      0.67      0.63      1301

    accuracy                           0.72      4839
   macro avg       0.71      0.72      0.72      4839
weighted avg       0.72      0.72      0.72      4839

EmoBERT model stored successfully.


In [2]:
# Pseudo-labeler by EmoBert

import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

def preprocess_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '') 
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    
    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025  + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class EmoBERTPureClassifier(nn.Module):
    def __init__(self, model_name='bhadresh-savani/bert-base-uncased-emotion', num_classes=3):
        super(EmoBERTPureClassifier, self).__init__()
        self.emobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.emobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.emobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

class PseudoLabelDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df_2025 = pd.read_csv("/kaggle/input/datasets/andryjosephtumacole/latest/2025.csv")
    df_2025['cleaned_text'] = df_2025['text'].apply(preprocess_tweet)

    model_name = 'bhadresh-savani/bert-base-uncased-emotion'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = EmoBERTPureClassifier(model_name=model_name, num_classes=3)
    model.load_state_dict(torch.load("emobert_model.pt", map_location=device))
    model.to(device)
    model.eval()

    inference_loader = DataLoader(PseudoLabelDataset(df_2025['cleaned_text'].values, tokenizer), batch_size=64, shuffle=False)

    all_preds, all_confidences = [], []
    with torch.no_grad():
        for batch in tqdm(inference_loader, desc="Pseudo-labeling"):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            probabilities = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probabilities, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

    df_2025['pseudo_label'] = [{0: 'Negative', 1: 'Neutral', 2: 'Positive'}[p] for p in all_preds]
    df_2025['confidence'] = all_confidences

    # Filter by 80% confidence
    df_filtered = df_2025[df_2025['confidence'] >= 0.80].copy()
    
    # Stratified downsampling to balance the dataset
    target_size = df_filtered['pseudo_label'].value_counts().min()
    df_balanced = df_filtered.groupby('pseudo_label', group_keys=False).apply(lambda x: x.sample(n=target_size, random_state=42))
    
    df_balanced.to_csv("2025_emobert_pseudo_balanced.csv", index=False)
    print(f"\nSaved perfectly balanced 2025 dataset with {len(df_balanced)} rows to '2025_emobert_pseudo_balanced.csv'")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Pseudo-labeling: 100%|██████████| 3401/3401 [27:39<00:00,  2.05it/s]
/tmp/ipykernel_57/1719349646.py:143: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df_filtered.groupby('pseudo_label', group_keys=False).apply(lambda x: x.sample(n=target_size, random_state=42))



Saved perfectly balanced 2025 dataset with 28971 rows to '2025_emobert_pseudo_balanced.csv'


In [3]:
# EmoBert Training with PseudoLabeled

import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def preprocess_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '') 
    
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    
    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025  + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class PHTweetElectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class EmoBERTPureClassifier(nn.Module):
    def __init__(self, model_name='bhadresh-savani/bert-base-uncased-emotion', num_classes=3):
        super(EmoBERTPureClassifier, self).__init__()
        self.emobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.emobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.emobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    df = pd.read_csv("2025_emobert_pseudo_balanced.csv")
    
    label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    df['label'] = df['pseudo_label'].map(label_mapping)
    
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['cleaned_text'].values, df['label'].values, train_size=0.8, test_size=0.2, random_state=42
    )

    model_name = 'bhadresh-savani/bert-base-uncased-emotion'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = EmoBERTPureClassifier(model_name=model_name, num_classes=3).to(device)

    # Increased batch size slightly for the larger dataset
    train_loader = DataLoader(PHTweetElectionDataset(train_texts, train_labels, tokenizer), batch_size=32, shuffle=True)
    test_loader = DataLoader(PHTweetElectionDataset(test_texts, test_labels, tokenizer), batch_size=32, shuffle=False)

    optimizer = AdamW([
        {'params': model.emobert.parameters(), 'lr': 2e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4}
    ], weight_decay=0.01)

    epochs = 3
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * epochs), num_training_steps=len(train_loader) * epochs)

    class_weights_tensor = torch.tensor(compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = criterion(logits, batch['labels'].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            predictions.extend(torch.argmax(logits, dim=1).flatten().cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

    print(f"\n===== MODEL ACCURACY: {accuracy_score(true_labels, predictions):.4f} =====")
    print(classification_report(true_labels, predictions, target_names=['Negative', 'Neutral', 'Positive']))

    torch.save(model.state_dict(), "emobert_2025_pseudo_model.pt")
    print("Final 2025 Pseudo-labeled model stored successfully.")

if __name__ == "__main__":
    main()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3 | Loss: 0.2494
Epoch 2/3 | Loss: 0.0418
Epoch 3/3 | Loss: 0.0122

===== MODEL ACCURACY: 0.9924 =====
              precision    recall  f1-score   support

    Negative       1.00      0.99      0.99      1916
     Neutral       0.99      0.99      0.99      1912
    Positive       0.99      0.99      0.99      1967

    accuracy                           0.99      5795
   macro avg       0.99      0.99      0.99      5795
weighted avg       0.99      0.99      0.99      5795

Final 2025 Pseudo-labeled model stored successfully.


In [4]:
import re
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '')  
    SENATORS_2025 = [
    "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
    "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
    "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
    "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
    "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
    "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
    "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
    "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
    "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
    "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
    "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
    "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
    "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
    "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
    "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
    "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
    "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
    "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
    "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
    "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
    "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
    "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
    "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
    "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
    "ping lacson", "sixto lagare", "alex lague", "raul lambino",
    "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
    "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
    "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
    "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
    "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
    "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
    "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
    "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
    "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
    "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
    "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
    "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
    "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
    "kuya wily red", "randy red", "randy restum", "willie wil revillame",
    "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
    "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
    "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
    "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
    "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
    "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
    "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    
    CANDIDATES_LIST = " ".join(SENATORS_2025 + CANDIDATES_LIST_2022).split(" ")
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    return text

class EmoBERTPureClassifier(nn.Module):
    def __init__(self, model_name='bhadresh-savani/bert-base-uncased-emotion', num_classes=3):
        super(EmoBERTPureClassifier, self).__init__()
        self.emobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.emobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.emobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_name = 'bhadresh-savani/bert-base-uncased-emotion'
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = EmoBERTPureClassifier(model_name=model_name, num_classes=3)
model.load_state_dict(torch.load("emobert_model.pt", map_location=device))
model.to(device)
model.eval() 

def predict_tweets(tweet_list):
    label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    results = []
    
    for raw_tweet in tweet_list:
        cleaned_tweet = preprocess_tweet(raw_tweet)
        
        encoding = tokenizer(
            cleaned_tweet,
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            probabilities = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(logits, dim=1).item()
            confidence = probabilities[0][pred_class].item()
            
        results.append({
            "raw": raw_tweet,
            "prediction": label_mapping[pred_class],
            "confidence": f"{confidence * 100:.2f}%"
        })
        
    return results

sample_tweets = [
    "Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸",
    "Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.",
    "Abangan na lang natin ang official count at huwag mag-away away.",
    "BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident",
    "I'm still undecided on who to vote for, need to read more platforms.",
    'Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble Islam faith. His actions do not speak of the sentiments, beliefs and behavior of peace-loving Muslims. His religion should not be an issue here, HIS BAD ATTITUDE AND POLITICAL DECISIONS ARE!',
    "MABUTI NGA SA ABS-CBN.  Sobrang mukhang pera talaga yang business establishment na yan."
]

print("\nRunning Inference on Sample Tweets...")
predictions = predict_tweets(sample_tweets)

for res in predictions:
    print(f"\nTweet: {res['raw']}")
    print(f"Predicted Sentiment: {res['prediction']} ({res['confidence']} confidence)")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Inference on Sample Tweets...

Tweet: Sana talaga manalo si Leni sa darating na eleksyon! #LeniRobredo2022 🌸
Predicted Sentiment: Positive (61.61% confidence)

Tweet: Grabe ang dumi ng pulitika ngayon, panay siraan lang naman sila.
Predicted Sentiment: Negative (45.04% confidence)

Tweet: Abangan na lang natin ang official count at huwag mag-away away.
Predicted Sentiment: Neutral (79.31% confidence)

Tweet: BBM pa rin solid tayo dito!! ❤️✌️ #BBMIsMyPresident
Predicted Sentiment: Neutral (49.36% confidence)

Tweet: I'm still undecided on who to vote for, need to read more platforms.
Predicted Sentiment: Neutral (95.00% confidence)

Tweet: Someone sent this to me and when I looked it up sa FB, I’m amazed at the comments from Muslim brothers and sisters.  They are all one in saying that “Islam should never be used to threaten those who disagree with someone’s political beliefs.”  It’s clear that ROBIN PADILLA is not the embodiment of the principles and teachings of the noble Isl